In [2]:
import duckdb

con = duckdb.connect("../data/processed/complaints.duckdb")

con.execute("""
    CREATE OR REPLACE TABLE complaints AS
    SELECT * FROM read_parquet('../data/processed/complaints_clean.parquet')
""")

con.execute("SELECT COUNT(*) AS total_rows FROM complaints").df()

,total_rows
0,99709


In [3]:
#How m,any complaints per year?
con.execute("""
    SELECT year, COUNT(*) AS complaints
    FROM complaints
    GROUP BY year
    ORDER BY year
""").df()

,year,complaints
0,2023,3
1,2024,13589
2,2025,38210
3,2026,47907


Note that 2023 data starts from december and 2026 data is till Nov-4. So don't take the count as huge drop

In [4]:
#which 10 companies get the most complaints?
con.execute("""
    SELECT company, COUNT(*) AS complaints
    FROM complaints
    GROUP BY company
    ORDER BY complaints DESC
    LIMIT 10
""").df()

,company,complaints
0,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",31123
1,"EQUIFAX, INC.",28922
2,EXPERIAN INFORMATION SOLUTIONS INC.,26646
3,"BLOCK, INC.",899
4,CAPITAL ONE FINANCIAL CORPORATION,535
5,RESURGENT CAPITAL SERVICES L.P.,485
6,LEXISNEXIS,436
7,ENCORE CAPITAL GROUP INC.,386
8,CL HOLDINGS LLC,361
9,"PORTFOLIO RECOVERY ASSOCIATES, LLC",343


In [5]:
#what share of complaints is each product?
con.execute("""
    SELECT product,
           COUNT(*) AS complaints,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_total
    FROM complaints
    GROUP BY product
    ORDER BY complaints DESC
""").df()

,product,complaints,pct_of_total
0,Credit reporting or other personal consumer re...,88510,88.8
1,Debt collection,5722,5.7
2,Credit card,1330,1.3
3,Checking or savings account,1297,1.3
4,"Money transfer, virtual currency, or money ser...",1114,1.1
5,Mortgage,448,0.4
6,Vehicle loan or lease,434,0.4
7,Student loan,306,0.3
8,"Payday loan, title loan, personal loan, or adv...",275,0.3
9,Prepaid card,178,0.2


In [6]:
#How do companies respond?
con.execute("""
    SELECT company_response_to_consumer AS response,
           COUNT(*) AS complaints
    FROM complaints
    GROUP BY response
    ORDER BY complaints DESC
""").df()

,response,complaints
0,Closed with explanation,59821
1,Closed with non-monetary relief,34748
2,In progress,4550
3,Closed with monetary relief,370
4,Untimely response,220


In [7]:
#how many complaints are credit reporting vs everything else?
con.execute("""
    SELECT is_credit_reporting,
           COUNT(*) AS complaints,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_total
    FROM complaints
    GROUP BY is_credit_reporting
""").df()

,is_credit_reporting,complaints,pct_of_total
0,False,11199,11.2
1,True,88510,88.8


Credit reporting is about 89% of all complaints, so I will analyze it separately from other products.

In [8]:
con.close()